# Rung 47 — `C_epochs`: is rung 42's +0.0402 the CORPUS or the EPOCHS?

**The ladder.** rung 21 designed arm `C_epochs` (single variable `--num_train_epochs`, off
A2), VRAM-probed it in `RESULTS_vram_C_epochs.json`, and never trained it. Rung 42 then
changed the corpus *and* ran two extra epochs, won **+0.0402**, and shipped. At the matched
epoch (ep3 vs ep3) rung 42's corpus **loses 0.0079**; the whole gain is at epoch 4, which the
control never ran. Rung 47 finally runs A2's own corpus to 5 epochs, so `ep4 − ep4` has the
corpus as its only difference.

**Where it runs.** UNAM, GPU 1 (`orena-train`). The arm trains on GPU 0 in `tmux leo-rung47`
— see `~/storage/rung47/OWNER.md`. This notebook must never touch GPU 0.

## What each epoch is for

| epoch | step | role |
|---|---|---|
| 1, 2 | 901, 1802 | trajectory — scored only when free |
| **3** | **2703** | 🔑 **the control.** Must reproduce A2's **0.6342** within ±0.01 |
| **4** | **3604** | 🎯 **the answer.** Against rung 42's ep4 **0.6744** |
| 5 | 4505 | in rung 42 this epoch already fell |

## 🔴 ep3 controls THREE things at once, not one

A2's 0.6342 came off a RunPod pod. Between it and this rung: `transformers` 4.57 → **5.12.1**,
frames from decord-off-`.mp4` → **JPEG q95 cache** (UNAM has every frame and zero `.mp4`, so
the cache is forced), and A100/L40S → **RTX 6000 Ada**. A green ep3 bounds all three jointly.
A red ep3 does **not** say which one moved.

## 🟢 The read that survives a red control — difference-in-differences

`47_ep4 − 42_ep4` crosses the two stacks. The **epoch effect inside each arm** does not:

    rung 42, merged corpus:  ep4 − ep3 = 0.6744 − 0.6262 = +0.0482
    rung 47, A2's corpus:    ep4 − ep3 = (this notebook)

If rung 47 also rises ≈ +0.048, the epoch effect is corpus-independent ⇒ rung 42's advantage
was **epochs**. That is the primary read. Everything else is reported beside it.

## 🔴 A2's per-question archive no longer exists

It lived on the pod (`repo_rodri/.../21_lr_2e4_v1/ep3_full/`) and is in neither S3 nor UNAM
(searched 2026-08-18). **The ep3 control is therefore a SCALAR check with no paired CI.**
Rung 42's archive *is* in S3 (`evidence_42/ep*_full/`, all five epochs) and is fetched below,
before the GPU.

In [ ]:
# --- bootstrap ---------------------------------------------------------------------
import json, logging, os, sys, time
from pathlib import Path
import pandas as pd

# 🔴 HF_HOME before the offline flags mean anything, and before any HF import.
# TWO caches exist on this box with DIFFERENT models: ~/.cache/huggingface (12 GB) and
# ~/storage/hf_cache (80 GB). The judge (Qwen/Qwen3-4B) and the Qwen3-VL-8B base are in
# the 80 GB one. Verified offline 2026-08-18.
os.environ.setdefault("HF_HOME", "/home/uaq_user/storage/hf_cache")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

# 🔴 `swift` is shelled out to for the merge, and a papermill kernel does NOT inherit the
# env's bin/ on PATH — it must be THIS interpreter's bin (rung 39's scar). On top of that
# the env is a `conda create --clone`, whose console scripts kept gen36's shebang until
# they were sed-repaired on 2026-08-18; log what actually resolves.
_envbin = str(Path(sys.executable).parent)
if _envbin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _envbin + os.pathsep + os.environ.get("PATH", "")

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s",
                    datefmt="%H:%M:%S")

REPO = "/home/uaq_user/storage/repo_leo"
sys.path.insert(0, f"{REPO}/experiments/45-gen36-data-and-reg/_tools")   # eval_arm45
sys.path.insert(0, f"{REPO}/experiments/47-epochs-vs-corpus/_tools")     # eval_arm47
from eval_arm45 import ensure_paths
ensure_paths(REPO)
import eval_arm47 as E

import transformers
print("python  :", sys.executable)
print("swift   :", (Path(_envbin) / "swift").exists(), "->", Path(_envbin) / "swift")
print("tfmrs   :", transformers.__version__, "(A2 was scored on 4.57 — that is ep3's job)")
print("HF_HOME :", os.environ["HF_HOME"])

In [ ]:
# --- parameters (RAW LITERALS ONLY — papermill injects BELOW this cell) -------------
SMOKE   = True          # True -> 40 questions on the FIRST epoch only. Full: -p SMOKE False
EPOCHS  = [3]           # [3] while ep4 is still training; [3, 4] once it lands; add 5 at the end
N_BOOT  = 4000
KEEP_MERGED = False     # a merged 8B is ~16 GB and this is a shared box

In [ ]:
# --- derived (MUST live BELOW the parameters cell — the rung-16 papermill trap) -----
cfg = E.Rung47Config(
    repo_root=REPO,
    epochs=tuple(EPOCHS),
    n_boot=N_BOOT,
    keep_merged=KEEP_MERGED,
    smoke=SMOKE,
)
EXP = Path(REPO) / "experiments" / "47-epochs-vs-corpus"
CTRL_DIR = cfg.run_dir / "controls"          # rung 42's archive lands here

CKPT_ROOT = E.resolve_ckpt_root(cfg)         # RAISES on 0 or >1 v0-* dirs
print("arm      :", E.RUN)
print("ckpt_root:", CKPT_ROOT)
print("epochs   :", EPOCHS, "-> steps", [E.EPOCH_STEPS[e] for e in EPOCHS])

In [ ]:
# --- GATES BEFORE THE GPU (all RAISE — RULES §7) -----------------------------------
# 1. the single-variable promise, from the launcher's own diff_vs_A2.json
print("single variable:", E.assert_single_variable(cfg.run_dir))

# 2. the run moved weights. rc=0 is NOT evidence — AdamW's decoupled weight decay moves
#    every tensor at zero gradient, so only grad_norm can tell a run from a no-op.
#    ⚠️ full.log is block-buffered and frozen at 44,854 bytes; it is not read here.
tr = E.assert_run_moved_weights(CKPT_ROOT)
print(f"train    : {tr['n_steps_logged']} steps logged, epoch {tr['last_epoch_logged']:.2f}, "
      f"loss {tr['loss_first']:.3f} -> {tr['loss_last']:.3f}")
print(f"token_acc: {tr['token_acc_first']:.3f} -> {tr['token_acc_last']:.3f}  "
      f"(TRAIN, not a result — the held-out curve is what says memorisation)")

# 3. every epoch we intend to score is fully written
for e in EPOCHS:
    print("checkpoint ok:", E.assert_checkpoint_complete(CKPT_ROOT, e))

In [ ]:
# --- the eval set, and that every frame it needs is already cached ------------------
# Rung 42's 8 held-out videos / 1,283 questions, with rung 42's own three asserts re-run:
# no leak between the two sides, no video in neither, and the count has not drifted.
split = E.held_out_split(cfg.to_eval_config(EPOCHS[0]))
print(f"held out : {len(split['held_videos'])} videos / {split['n_held']} questions "
      f"({split['n_heico_videos']} heico videos)")

# CachedFrameProvider RAISES on a miss rather than falling back — correct, but the miss
# would surface with the model already loaded. This walks the same identity keys with
# os.path.exists and costs seconds.
print(E.assert_cache_covers(cfg.to_eval_config(EPOCHS[0]), split["held_items"]))
HELD_QIDS = split["held_qids"]

In [ ]:
# --- PRE-FLIGHT: the judge must resolve offline, BEFORE anything expensive ---------
# run_baseline loads the judge only AFTER the whole inference pass, so a missing cache
# fails ~30 min in with everything already paid for. RAISES (RULES §7).
from transformers import AutoTokenizer
from frame.config import BaselineConfig

_judge = BaselineConfig().judge_model
try:
    AutoTokenizer.from_pretrained(_judge)
except Exception as exc:
    raise AssertionError(
        f"JUDGE GATE FAILED: {_judge!r} does not resolve offline ({type(exc).__name__}). "
        f"HF_HOME={os.environ.get('HF_HOME')!r}. Fix the env — do NOT disable the offline "
        "flags, the whole deployment story is offline."
    ) from exc
print(f"OK judge gate: {_judge} resolves from {os.environ['HF_HOME']}")

In [ ]:
# --- the CONTROL ARCHIVE, fetched before the GPU -----------------------------------
# Rung 42's per-question answers. Without them the comparison this rung exists for
# degrades to two scalars with no CI — and that is discovered AFTER the eval, not before
# (rung 45's scar). ~155 KB per epoch.
# 🔴 UNAM has no boto3 and no S3 credentials: here this only VERIFIES (1,283 rows, 800
# heico, the four columns the CI needs). The files are pushed from the laptop.
CTRL42 = E.ensure_control_42(CTRL_DIR, epochs=(3, 4, 5))
for e, p in CTRL42.items():
    print(f"rung 42 ep{e}: {p}  ({p.stat().st_size:,} bytes)")

print("\nrung 42's archived bucket_mean by epoch:",
      "  ".join(f"ep{e} {v:.4f}" for e, v in E.CONTROL_42["bucket_mean"].items()))
print(f"A2 ep3 (scalars only, archive is GONE): {E.CONTROL_A2_EP3['bucket_mean']:.4f}")

In [ ]:
# --- the SWEEP: merge -> eval on the 1,283 -> reclaim the 16 GB --------------------
# One epoch at a time. `score_epoch` reuses an existing results.csv, so re-running this
# notebook to add ep4 costs ZERO GPU on ep3 — that is what makes the sweep incremental.
ARM = {}
for e in EPOCHS:
    t0 = time.perf_counter()
    print(f"=== epoch {e} (checkpoint-{E.EPOCH_STEPS[e]}) ===")
    ARM[e] = E.score_epoch(cfg, e, split)
    c = ARM[e]["cells"]
    print(f"    bucket_mean {c['bucket_mean']:.4f}  acc_ID {c['acc_ID']:.4f}  "
          f"acc_heico {c['acc_OOD']:.4f}   [{time.perf_counter() - t0:.0f}s]")

In [ ]:
# --- 🔑 THE CONTROL VERDICT: did the stack reproduce A2? ---------------------------
# Declared before the number (RULES §S4): GREEN inside ±0.01 of 0.6342.
if 3 in ARM:
    v = E.control_verdict(ARM[3]["cells"]["bucket_mean"])
    print(f"rung 47 ep3 : {v['ep3_bucket_mean']:.4f}")
    print(f"A2      ep3 : {v['a2_ep3_bucket_mean']:.4f}   (archived scalar, no paired CI)")
    print(f"delta       : {v['delta']:+.4f}   tolerance ±{v['tolerance']}")
    print(f"\n{v['verdict']} — {v['means']}")
else:
    print("ep3 not in this sweep — no control, and therefore no verdict.")

In [ ]:
# --- 🎯 the per-epoch table, and the DiD --------------------------------------------
rows = []
for e in sorted(ARM):
    rows.append({"arm": "47_C_epochs (A2 corpus)", "epoch": e,
                 "ckpt": f"checkpoint-{ARM[e]['step']}",
                 **{k: round(v, 4) for k, v in ARM[e]["cells"].items()}})
for e, bm in E.CONTROL_42["bucket_mean"].items():
    rows.append({"arm": "42_merged (archived)", "epoch": e, "ckpt": "", "bucket_mean": bm})
rows.append({"arm": "A2 ep3 (archived)", "epoch": 3, "ckpt": "checkpoint-2703",
             "bucket_mean": E.CONTROL_A2_EP3["bucket_mean"],
             "acc_ID": E.CONTROL_A2_EP3["acc_ID"], "acc_OOD": E.CONTROL_A2_EP3["acc_OOD"]})
tbl = pd.DataFrame(rows)
cols = [c for c in ["arm", "epoch", "ckpt", "bucket_mean", "acc_ID", "acc_OOD",
                    "aggregation_ID", "object_recognition_ID",
                    "aggregation_OOD", "object_recognition_OOD"] if c in tbl.columns]
print(tbl[cols].to_string(index=False))

if 3 in ARM and 4 in ARM:
    d = E.did_verdict(ARM[3]["cells"]["bucket_mean"], ARM[4]["cells"]["bucket_mean"])
    print(f"\n--- difference-in-differences (stack-free: both sides internal to one arm) ---")
    print(f"rung 47 (A2 corpus)     ep4 − ep3 = {d['d47_ep4_minus_ep3']:+.4f}")
    print(f"rung 42 (merged corpus) ep4 − ep3 = {d['d42_ep4_minus_ep3']:+.4f}")
    print(f"DiD                               = {d['did']:+.4f}")
    print(f"\n{d['reads']}")
else:
    print("\nDiD needs BOTH ep3 and ep4 — not yet.")

In [ ]:
# --- class-balanced F1 on `fo_class` — MANDATORY before publishing (RULES §9b) -----
# `fo_class` accuracy is exact SET equality, so it is dominated by the head of a
# long-tailed distribution and cannot see a tail collapse. Rung 45's 27B emitted ZERO
# illegal tokens while macro-F1 halved: it collapsed onto `Clip`. Read the per_class
# table BESIDE the scalar.
from frame import ledger, metrics

gold = ledger.gold_from_frame_parquets(Path(cfg.data_root))
F1, f1rows = {}, []
if not SMOKE:
    for e in sorted(ARM):
        preds = metrics.predictions_frame(Path(cfg.eval_dir(e)) / ARM[e]["run_name"])
        preds = preds[preds["qID"].isin(HELD_QIDS)]
        res = pd.read_csv(ARM[e]["results_csv"])
        res = res[res["qID"].isin(HELD_QIDS)]
        F1[e] = metrics.class_f1_report(preds, gold, results_df=res, n_boot=2000)
        for cell in ("pooled", "ID", "OOD"):
            b = F1[e][cell]
            f1rows.append({"epoch": e, "cell": cell, "n": b["n"], "illegal": b["n_illegal"],
                           "macro_f1": round(b["macro_f1"], 4),
                           "exact_set_acc": round(b["exact_set_acc"], 4),
                           "ci": f"[{b['ci_low']:.3f}, {b['ci_high']:.3f}]"})
    f1_df = pd.DataFrame(f1rows)
    print(f1_df.to_string(index=False))

    _sel = max(ARM)
    _per = pd.DataFrame(F1[_sel]["ID"]["per_class"]).T
    print(f"\n--- per class, ID, epoch {_sel} (the tail is the point) ---")
    print("(no fo_class x ID rows)" if _per.empty else
          _per[["n_gold", "recall", "precision", "f1"]]
          .sort_values("n_gold", ascending=False).round(3).to_string())
    # 🔴 an illegal class token does not score 0 — FOType.from_name() RAISES (RULES §8b)
    _ill = f1_df[f1_df.illegal > 0]
    if len(_ill):
        print("\n🔴 ILLEGAL fo_class tokens emitted (RULES §8b):")
        print(_ill.to_string(index=False))
else:
    print("SMOKE — F1 skipped")

In [ ]:
# --- the PAIRED, VIDEO-CLUSTERED CI vs rung 42's SAME epoch ------------------------
# 🔴 Effective n is 8 VIDEOS, not 1,283 questions (RULES §13). Unclustered would be ~10x
# too narrow and would manufacture significance.
# 🔴 Two videos carry the whole heico side, so no `*_OOD` CI is READABLE. Every cell is
# computed anyway: RULES §S8 lets ANY cell veto, while only a pre-declared cell may win.
# 🔴 This comparison crosses two stacks (see the header). If ep3 came back RED, it is not
# a corpus comparison and must not be quoted as one.
ci_df = None
if not SMOKE:
    for e in sorted(ARM):
        if e not in CTRL42:
            continue
        ci_df = E.paired_ci_vs_42(ARM[e]["results_csv"], CTRL42[e], HELD_QIDS, n_boot=N_BOOT)
        d_bm = ARM[e]["cells"]["bucket_mean"] - E.CONTROL_42["bucket_mean"][e]
        print(f"\n=== rung 47 ep{e} MINUS rung 42 ep{e}, on the 8 held-out videos ===")
        print(ci_df.to_string(index=False))
        print(f"d(bucket_mean) = {d_bm:+.4f}")
        print("cells favouring rung 47 (CI excludes 0):",
              ci_df[(ci_df.excludes_zero) & (ci_df.delta > 0)]["cell"].tolist())
        print("cells favouring rung 42 (CI excludes 0, VETO):",
              ci_df[(ci_df.excludes_zero) & (ci_df.delta < 0)]["cell"].tolist())
    print("\nRULES §S4: below |d| = 0.01 nothing is readable. §S1: acting needs |d| ~ 0.03, "
          "and that is a team call with the cost stated, never automatic.")
else:
    print("SMOKE — no CI, no verdict, no selection")

In [ ]:
# --- persist: RESULTS.csv + the ledger (full sweeps with BOTH decisive epochs only) -
# 🔴 Deliberately gated on {3, 4}. A partial sweep would append rows whose `selected`
# column means "the best of the epochs that happened to be trained by then", and the
# next run would append DUPLICATES on top of them (rung 42's notebook concats).
if not SMOKE and {3, 4} <= set(ARM):
    ctrl = E.control_verdict(ARM[3]["cells"]["bucket_mean"])
    did = E.did_verdict(ARM[3]["cells"]["bucket_mean"], ARM[4]["cells"]["bucket_mean"])
    sel = max(ARM, key=lambda e: ARM[e]["cells"]["bucket_mean"])
    out_rows = []
    for e in sorted(ARM):
        c = ARM[e]["cells"]
        res = pd.read_csv(ARM[e]["results_csv"])
        res = res[res["qID"].isin(HELD_QIDS)]
        row = {
            "run": E.RUN, "arm": "47_C_epochs", "epoch": e,
            "checkpoint": f"checkpoint-{ARM[e]['step']}",
            "eval_set": "8 held-out videos (1283 q)", "n_eval": len(HELD_QIDS),
            "n_videos": 8, "n_heico_videos": 2,
            # A2's corpus promoted NO test video, so unlike rung 42 the OOD label here
            # means what it means everywhere else in the repo.
            "ood_procedure_in_train": False,
            "baseline_run": "42_merged_v1", "baseline_epoch": e,
            "selected": e == sel,
            **{k: c[k] for k in ("bucket_mean", "acc_ID", "acc_OOD", "margin_ID", "margin_OOD",
                                 "aggregation_ID", "object_recognition_ID",
                                 "aggregation_OOD", "object_recognition_OOD")},
            "macro_f1_ID": F1[e]["ID"]["macro_f1"], "macro_f1_OOD": F1[e]["OOD"]["macro_f1"],
            "d_bucket_mean_vs_42_same_epoch":
                c["bucket_mean"] - E.CONTROL_42["bucket_mean"].get(e, float("nan")),
            "stack_control": ctrl["verdict"],
        }
        metrics.assert_class_f1_reported(row, results_df=res)      # RULES §9b — RAISES
        out_rows.append(row)

    df = pd.DataFrame(out_rows)
    out = EXP / "RESULTS.csv"
    if out.exists():
        df = pd.concat([pd.read_csv(out), df], ignore_index=True)
    df.to_csv(out, index=False)
    f1_df.to_csv(EXP / "RESULTS_class_f1.csv", index=False)
    if ci_df is not None:
        ci_df.to_csv(EXP / "RESULTS_paired_ci.csv", index=False)
    (EXP / "RESULTS_epochs_vs_corpus.json").write_text(json.dumps({
        "question": ("rung 42's +0.0402 — corpus or epochs? Same corpus as A2 (14,415), "
                     "5 epochs, so ep4 vs ep4 has the corpus as its only difference."),
        "single_variable": E.SINGLE_VARIABLE,
        "eval_set": {"videos": sorted("/".join(v) for v in split["held_videos"]),
                     "n_questions": len(HELD_QIDS), "n_heico_videos": 2},
        "stack_control": ctrl,
        "difference_in_differences": did,
        "stack_confounds_vs_A2": ["transformers 4.57 -> 5.12.1",
                                  "FrameProvider (decord/mp4) -> CachedFrameProvider (JPEG q95)",
                                  "A100/L40S -> RTX 6000 Ada"],
        "a2_per_question_archive": "GONE — searched S3 and UNAM 2026-08-18; control is scalar",
        "per_epoch": {str(e): ARM[e]["cells"] for e in sorted(ARM)},
    }, indent=2), encoding="utf-8")
    print("wrote:", out, EXP / "RESULTS_class_f1.csv", EXP / "RESULTS_epochs_vs_corpus.json")
else:
    print("not persisted — needs a full (non-SMOKE) sweep containing BOTH ep3 and ep4")

In [ ]:
# --- eyeball: what did it actually SAY? (user rule: examples on every run) ---------
if ARM:
    e = max(ARM)
    res = pd.read_csv(ARM[e]["results_csv"])
    res = res[res["qID"].isin(HELD_QIDS)]
    preds = metrics.predictions_frame(Path(cfg.eval_dir(e)) / ARM[e]["run_name"])
    fo = res[res.answer_format == "fo_class"].merge(preds, on="qID")
    print(f"--- epoch {e}: fo_class misses (identity, not cardinality, is the usual one) ---")
    print(fo[fo.correctness == 0].head(12)[["qID", "prediction"]].to_string(index=False))

    g = gold[gold.qID.isin(HELD_QIDS)].copy()
    g["template"] = g["question"].map(metrics.template_of)
    clips = g[g.template == "How many Clips appear in this frame? Please provide a number."]
    look = clips.merge(preds, on="qID")
    if len(look):
        look["gold_n"] = look["answer"].map(metrics.read_count)
        look["pred_n"] = look["prediction"].map(metrics.read_count)
        print("\n--- predicted-vs-gold crosstab (Clips), held-out videos ---")
        print(pd.crosstab(look["gold_n"], look["pred_n"]).to_string())